# Day 29 — Data validation & schemas (pandera/pydantic)
Objectives:
- Define dataframes schemas with pandera.
- Validate inputs and raise clear errors.
- Use in pipelines to enforce contracts.

<!-- BEGIN BEGINNER NOTEBOOK DEEP DIVE -->
## How to use this notebook

This is the editable learner artifact for `python-29`. Read
`python/ds-60day/companion-guides/day29_data_validation_schemas.md` first, then work here with the **Python (ds60sqlpy)**
kernel. Restart the kernel and run from top to bottom so an earlier
hidden value cannot make later code appear correct.

For every example: (1) write a prediction, (2) run the cell,
(3) compare the exact value, type, shape, rows, or side effect with
the stated observation, and (4) explain one mismatch before moving
on. For every exercise, use its dedicated work cell and include a
real assertion or bounded inspection. The official solution stays
closed until you have a tested attempt.

The notebook is deliberately offline after course setup. Do not add
`%pip`, credentials, absolute developer paths, or shell-specific
setup here. If an import fails, use the repository doctor and the
catalog from a terminal rather than changing only this kernel.

## Core mental model

A data schema describes more than column names: required presence,
dtypes, nullability, ranges, allowed categories, uniqueness, and
cross-field rules. Derive constraints from domain/business meaning and
declared interfaces, not merely from values observed in one sample.

Validation should sit immediately before a boundary that depends on the
contract, especially saving or handing data to a model. Coercion changes
compatible representations; validation checks the resulting state.
Treat a failure as blocked output with actionable evidence. Test one
valid fixture and one deliberately invalid fixture for every important
rule.

### Vocabulary

- **schema:** an executable description of data fields and constraints.
- **constraint:** a rule values or records must satisfy.
- **nullability:** whether absence is allowed.
- **coercion:** conversion toward a declared representation.
- **validation:** checking actual data against a contract.
- **failure report:** structured evidence identifying violated rules and locations.

## Syntax anatomy

In Pandera, `pa.Column(float, checks=pa.Check.ge(0), nullable=False)`
states representation, a non-negative constraint, and missing policy.
`schema.validate(frame, lazy=True)` collects multiple failures before
raising a `SchemaErrors` report. Coercion, if enabled, occurs before
checks and must not be mistaken for silently repairing invalid meaning.

### Worked example 1 — Validate a small table against explicit rules

A good fixture makes the accepted contract easy to see. Before running the next cell, predict its final displayed
value and identify the line responsible for every intermediate.

In [ ]:
import pandas as pd
import pandera.pandas as pa

schema = pa.DataFrameSchema({
    "quantity": pa.Column(int, checks=pa.Check.ge(0)),
    "price": pa.Column(float, checks=pa.Check.ge(0), coerce=True),
    "status": pa.Column(str, checks=pa.Check.isin(["open", "closed"])),
})
valid = pd.DataFrame({
    "quantity": [1, 0],
    "price": [2.5, 0.0],
    "status": ["open", "closed"],
})
checked = schema.validate(valid)
checked.to_dict("records")

**Expected observation:** The two records are returned after successful validation; `price` follows the declared floating representation.

If your result differs, compare inputs and types before rerunning.
Then explain the example from input to evidence in your own words.

### Worked example 2 — Show that validation guards persistence

Only data returned from validation should continue to a write boundary. Predict first; then run the next cell.

In [ ]:
def validated_for_output(frame: pd.DataFrame) -> pd.DataFrame:
    checked = schema.validate(frame, lazy=True)
    return checked.copy()

output_ready = validated_for_output(valid)
(len(output_ready), output_ready["quantity"].min())

**Expected observation:** `(2, 0)`. An invalid frame would raise before any output-writing code is reached.

## Debugging clinic

When evidence differs from your prediction, use this order:

1. Write the business meaning and exact boundary values before encoding a check.
2. Separate cleaning/coercion from validation and count values changed by coercion.
3. Use lazy validation during diagnosis to see multiple rule failures together.
4. Do not catch and discard schema errors or save before validation.

**Alternative to compare:** Use Pydantic for individual Python records/configuration and Pandera for DataFrame-wide column/index checks; database constraints protect stored relational data.

**Boundary to test:** Nulls, infinities, dtype coercion, duplicate composite keys, unexpected categories, empty frames, and cross-column rules need fixtures.

Do not move on merely because the cell runs. Explain which object,
branch, axis, row, or resource changed and why.

In [ ]:
import pandas as pd
import pandera.pandas as pa
from pandera.typing import DataFrame, Series

class InputSchema(pa.DataFrameModel):
    a: Series[int] = pa.Field(ge=0)
    b: Series[float] = pa.Field(nullable=False)

@pa.check_types
def add_ab(df: DataFrame[InputSchema]) -> pd.Series:
    return df['a'] + df['b']

df = pd.DataFrame({'a':[1,2,3], 'b':[0.5, 1.5, 2.0]})
add_ab(df)


## Exercises and progressive hints

Each item is a complete mini-contract. Before writing code, copy its input,
expected behavior, constraints, and verification into your work cell. A
result is not complete merely because it “looks right”; run the stated
assertion or inspection and explain what it proves.

1. Extend a Pandera schema with documented inclusive/exclusive ranges, allowed categories, nullability, and key uniqueness. **Constraints:** derive each rule from written business meaning and include exact boundary values.
   **Verify:** one valid fixture passes and a separate invalid fixture for each rule fails with the expected column/check evidence.

2. Validate the cleaned Day 18 DataFrame immediately before saving. **Sequence:** clean → validate returned frame → write only validated output → re-read/reconcile.
   **Expected behavior:** a deliberately invalid fixture blocks the write and preserves a useful failure report. **Constraint:** do not catch and discard `SchemaError`/`SchemaErrors`.
   **Verify:** Use a temporary output path to prove valid data writes/reloads, while an invalid fixture raises before the file exists.

### Additional mastery practice

Translate business rules into executable boundary contracts and prove both acceptance and informative failure before persisting data.

Continue with five new exercises. Record each prediction before running
code; these extend rather than replace the original practice above.

3. **Prediction:** Predict the difference between coercing a value to the schema dtype and validating it without coercion.
   **Progressive hint:** Coercion transforms compatible representations; validation checks a state.
   **Verify:** Validate numeric text with/without coercion and assert the coerced dtype/value versus the strict failure; record what changed rather than calling it mere validation.
4. **Tracing:** Trace a row through cleaning, schema validation, and saving. At which boundary must failure stop the write?
   **Progressive hint:** Validate the actual final frame immediately before persistence.
   **Verify:** Use a temporary path and event log; assert valid rows reach save only after validation and invalid rows raise before any file/write event.
5. **Implementation:** Create a Pandera schema with non-negative quantity, finite price, and an allowed status set, then validate one good fixture.
   **Progressive hint:** Derive constraints from stated business rules, not observed values alone.
   **Verify:** Assert one good fixture returns unchanged meaning, then independently fail negative quantity, nonfinite price, and unknown status with named checks.
6. **Debugging:** Repair a pipeline that writes output before validating it or catches and discards every schema error.
   **Progressive hint:** Validation failure is a blocked output, not a warning-only event.
   **Verify:** Reorder the pipeline and assert invalid data leaves no output file; retain and inspect the schema failure instead of converting it to a warning.
7. **Edge case and explanation:** Create invalid fixtures for null, range, category, duplicate-key, and dtype rules; use lazy validation to inspect multiple failures.
   **Progressive hint:** One failure fixture per rule makes contract coverage auditable.
   **Verify:** Run lazy validation on fixtures violating all five rule families and assert the failure cases include each expected column/check category.

Before opening the reference solution, write one sentence explaining
which contract or mental model each result confirms.

### Practice 1 — prediction, attempt, and evidence

**Contract reminder:** Extend a Pandera schema with documented inclusive/exclusive ranges, allowed categories, nullability, and key uniqueness. **Constraints:** derive each rule from written business meaning and include exact boundary values. **Verify:** one valid fixture passes and a separate invalid fixture for each rule fails with the expected column/check evidence.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 1 — your work
# Short contract: Extend a Pandera schema with documented inclusive/exclusive ranges, allowed categories, nullability, and key uniqueness. derive each rule from written business meaning and inclu...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 2 — prediction, attempt, and evidence

**Contract reminder:** Validate the cleaned Day 18 DataFrame immediately before saving. **Sequence:** clean → validate returned frame → write only validated output → re-read/reconcile. **Expected behavior:** a deliberately invalid fixture blocks the write and preserves a useful failure report. **Constraint:** do not catch and discard `SchemaError`/`SchemaErrors`. **Verify:** Use a temporary output path to prove valid data writes/reloads, while an invalid fixture raises before the file exists.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 2 — your work
# Short contract: Validate the cleaned Day 18 DataFrame immediately before saving. clean → validate returned frame → write only validated output → re-read/reconcile. a deliberately invalid fixtur...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 3 — prediction, attempt, and evidence

**Contract reminder:** **Prediction:** Predict the difference between coercing a value to the schema dtype and validating it without coercion. **Progressive hint:** Coercion transforms compatible representations; validation checks a state. **Verify:** Validate numeric text with/without coercion and assert the coerced dtype/value versus the strict failure; record what changed rather than calling it mere validation.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 3 — your work
# Short contract: Predict the difference between coercing a value to the schema dtype and validating it without coercion. Coercion transforms compatible representations; validation checks a state...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 4 — prediction, attempt, and evidence

**Contract reminder:** **Tracing:** Trace a row through cleaning, schema validation, and saving. At which boundary must failure stop the write? **Progressive hint:** Validate the actual final frame immediately before persistence. **Verify:** Use a temporary path and event log; assert valid rows reach save only after validation and invalid rows raise before any file/write event.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 4 — your work
# Short contract: Trace a row through cleaning, schema validation, and saving. At which boundary must failure stop the write? Validate the actual final frame immediately before persistence. Use a...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 5 — prediction, attempt, and evidence

**Contract reminder:** **Implementation:** Create a Pandera schema with non-negative quantity, finite price, and an allowed status set, then validate one good fixture. **Progressive hint:** Derive constraints from stated business rules, not observed values alone. **Verify:** Assert one good fixture returns unchanged meaning, then independently fail negative quantity, nonfinite price, and unknown status with named checks.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 5 — your work
# Short contract: Create a Pandera schema with non-negative quantity, finite price, and an allowed status set, then validate one good fixture. Derive constraints from stated business rules, not o...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 6 — prediction, attempt, and evidence

**Contract reminder:** **Debugging:** Repair a pipeline that writes output before validating it or catches and discards every schema error. **Progressive hint:** Validation failure is a blocked output, not a warning-only event. **Verify:** Reorder the pipeline and assert invalid data leaves no output file; retain and inspect the schema failure instead of converting it to a warning.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 6 — your work
# Short contract: Repair a pipeline that writes output before validating it or catches and discards every schema error. Validation failure is a blocked output, not a warning-only event. Reorder t...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 7 — prediction, attempt, and evidence

**Contract reminder:** **Edge case and explanation:** Create invalid fixtures for null, range, category, duplicate-key, and dtype rules; use lazy validation to inspect multiple failures. **Progressive hint:** One failure fixture per rule makes contract coverage auditable. **Verify:** Run lazy validation on fixtures violating all five rule families and assert the failure cases include each expected column/check category.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 7 — your work
# Short contract: Create invalid fixtures for null, range, category, duplicate-key, and dtype rules; use lazy validation to inspect multiple failures. One failure fixture per rule makes contract...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):
